# Portfolio Replication — Full Comparison
Each section calls a single function from `utils/`. All outputs (figures, pickles) are saved automatically.

**Run order:** Data Loader → Linear Models → Kalman → NN → Evaluate

In [1]:
import sys, pickle, logging
from pathlib import Path

ROOT = Path('..').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from utils import setup_logging
setup_logging(logging.INFO)

PICKLE_DIR = ROOT / 'results' / 'data' / 'picklefiles'

## 1 · Data Loader
Loads prices, computes weekly returns, constructs the **monster index** (50% HFRX, 25% MSCI World, 25% Global Agg Bond), and saves 5 diagnostic plots.

In [2]:
from utils.data_loader import run_data_loader

data = run_data_loader(
    filepath=ROOT / 'data' / 'Dataset3_PortfolioReplicaStrategy.xlsx'
)

13:44:28 | INFO     | utils.data_loader         — ============================================================
13:44:28 | INFO     | utils.data_loader         — DATA LOADER — START
13:44:28 | INFO     | utils.data_loader         — ============================================================
13:44:28 | INFO     | utils.data_loader         — Reading Excel file: C:\Users\giuli\Repositories\fintech-group-work\BusinessCase3\data\Dataset3_PortfolioReplicaStrategy.xlsx (sheet=0)
13:44:28 | INFO     | utils.data_loader         — Loaded 705 observations | 2007-10-23 → 2021-04-20
13:44:28 | INFO     | utils.data_loader         — Computing weekly returns from price levels
13:44:28 | INFO     | utils.data_loader         — Returns shape: (704, 15)
13:44:28 | INFO     | utils.data_loader         — Monster index composition: HFRXGL Index=50% | MXWO Index=25% | LEGATRUU Index=25%
13:44:28 | INFO     | utils.data_loader         — X shape: (704, 11) | y shape: (704,)
13:44:28 | INFO     | utils.data_loa

In [3]:
print('Prices  :', data['prices'].shape,  '|', data['prices'].index[0].date(), '→', data['prices'].index[-1].date())
print('Returns :', data['returns'].shape)
print('X       :', data['X'].shape,  '(futures features)')
print('y       :', data['y'].shape,  '(monster index)')
data['y'].describe().round(5)

Prices  : (705, 15) | 2007-10-23 → 2021-04-20
Returns : (704, 15)
X       : (704, 11) (futures features)
y       : (704,) (monster index)


count    704.00000
mean       0.00049
std        0.00878
min       -0.06694
25%       -0.00361
50%        0.00143
75%        0.00552
max        0.03377
Name: MonsterIndex, dtype: float64

## 2 · Linear Models
Five estimators on a shared rolling walk-forward loop.
Hyperparameters selected by TimeSeriesSplit CV on training data only.

| Model | Penalty | Notes |
|---|---|---|
| OLS | none | unconstrained baseline |
| Ridge | L2 | uniform shrinkage |
| LASSO | L1 | sparse, interpretable |
| ElasticNet | L1+L2 | combined |
| WOLS | — | exponential decay weights, λ selected by TE-minimising CV |

In [4]:
from utils.run_linear_models import run_linear_models

linear_data = run_linear_models(
    X=data['X'],
    y=data['y'],
    train_end_date='2018-12-31',
    window=52,
)

13:44:31 | INFO     | utils.run_linear_models   — ============================================================
13:44:31 | INFO     | utils.run_linear_models   — LINEAR MODELS — START
13:44:31 | INFO     | utils.run_linear_models   — ============================================================
13:44:31 | INFO     | utils.run_linear_models   — Train: 583 obs (2007-10-30 → 2018-12-31) | Test: 121 obs (2019-01-01 → 2021-04-20) | window=52
13:44:31 | INFO     | utils.run_linear_models   — Scaler fit on 583 training observations
13:44:31 | INFO     | utils.run_linear_models   — --------------------------------------------------
13:44:31 | INFO     | utils.run_linear_models   — Hyperparameter selection (training data only) …
13:44:31 | INFO     | utils.models              — Ridge CV best alpha = 1.00e+00
13:44:31 | INFO     | utils.models              — LASSO CV best alpha = 1.00e-04
13:44:31 | INFO     | utils.models              — ElasticNet CV best alpha = 1.00e-03 | l1_ratio = 0.50
13:44:

In [5]:
# Metrics table (test period)
linear_data['metrics_df'][[
    'ann_ret_replica', 'tracking_error',
    'information_ratio', 'correlation', 'sharpe_replica',
]].round(4)

,ann_ret_replica,tracking_error,information_ratio,correlation,sharpe_replica
model,,,,,
OLS,0.0908,0.0380,-0.5265,0.8484,1.4805
Ridge,0.1005,0.0419,-0.2441,0.8116,1.7002
LASSO,0.1002,0.0414,-0.2548,0.8166,1.6747
ElasticNet,0.0992,0.0445,-0.2593,0.7860,1.8740
WOLS,0.0914,0.0375,-0.5155,0.8523,1.4922


In [6]:
# Load pre-computed if already pickled and you want to skip re-running
linear_pkl = PICKLE_DIR / 'linear_results.pkl'
if linear_pkl.exists():
    with open(linear_pkl, 'rb') as fh:
        linear_data = pickle.load(fh)
    linear_results = linear_data['best_results']
    print('Linear models loaded:', list(linear_results.keys()))
else:
    linear_results = {}
    print('No linear model pickle — run the cell above first.')

Linear models loaded: ['OLS', 'Ridge', 'LASSO', 'ElasticNet', 'WOLS']


## 3 · Kalman Filter
Random-walk state-space model for time-varying portfolio weights.
Q and R estimated from training data (OLS residual variance + grid search).

In [7]:
from utils.run_kalman import run_kalman

kalman_data = run_kalman(
    X=data['X'],
    y=data['y'],
    train_end_date='2018-12-31',
)

13:44:37 | INFO     | utils.run_kalman          — ============================================================
13:44:37 | INFO     | utils.run_kalman          — KALMAN FILTER — START
13:44:37 | INFO     | utils.run_kalman          — ============================================================
13:44:37 | INFO     | utils.run_kalman          — Train: 583 obs (2007-10-30 → 2018-12-31) | Test: 121 obs
13:44:37 | INFO     | utils.run_kalman          — Estimating R …
13:44:37 | INFO     | utils.run_kalman          — Estimated R (observation noise) = 0.000014
13:44:37 | INFO     | utils.run_kalman          — Selecting q via grid search on training data …
13:44:37 | INFO     | utils.run_kalman          — Kalman Q grid search: best q = 1.42e-05  (mean squared innovation = 0.931811)
13:44:37 | INFO     | utils.run_kalman          — Parameters: R = 1.42e-05 | q = 1.42e-05
13:44:37 | INFO     | utils.run_kalman          — Filter initialised with OLS weights (warm start)
13:44:37 | INFO     | utils

In [8]:
import pandas as pd
pd.DataFrame([kalman_data['metrics']])[[
    'ann_ret_replica', 'tracking_error',
    'information_ratio', 'correlation', 'sharpe_replica',
]].round(4)

,ann_ret_replica,tracking_error,information_ratio,correlation,sharpe_replica
0,0.1029,0.0445,-0.1764,0.7865,1.6832


In [9]:
kalman_pkl = PICKLE_DIR / 'kalman_results.pkl'
if kalman_pkl.exists():
    with open(kalman_pkl, 'rb') as fh:
        kalman_data = pickle.load(fh)
    kalman_result = kalman_data['best_result']
    print('Kalman loaded | params:', {k: round(v,6) for k,v in kalman_data['params'].items() if k != 'Q'})
else:
    kalman_result = None
    print('No Kalman pickle — run the cell above first.')

Kalman loaded | params: {'R': 1.4e-05, 'q': np.float64(1.4e-05)}


## 4 · Neural Network
MLP and LSTM weight-generator networks.

**Loss**: MSE(replica, target) + λ·L1(weights)  
**Post-processing**: VaR scaling to respect UCITS 1M VaR(99%) ≤ 20%

In [10]:
from utils.run_nn import run_nn, DEFAULT_CONFIGS

nn_output = run_nn(
    X=data['X'],
    y=data['y'],
    configs=DEFAULT_CONFIGS,
    train_frac=0.60,
    val_frac=0.15,
    max_var_threshold=0.20,
)

13:44:38 | INFO     | utils.run_nn              — ============================================================
13:44:38 | INFO     | utils.run_nn              — NN EXPERIMENT — START
13:44:38 | INFO     | utils.run_nn              — ============================================================
13:44:38 | INFO     | utils.run_nn              — Data split: train=422 | val=106 | test=176 (total=704)
13:44:38 | INFO     | utils.run_nn              — --------------------------------------------------
13:44:38 | INFO     | utils.run_nn              — Config 1/4: MLP_w26_h64x32_l10.0
13:44:38 | INFO     | utils.run_nn              — Device: cpu
13:44:40 | INFO     | utils.run_nn              — Epoch    1/300 | train=0.000141 | val=0.000039 | patience 0/30
13:44:41 | INFO     | utils.run_nn              — Epoch   25/300 | train=0.000010 | val=0.000008 | patience 16/30
13:44:41 | INFO     | utils.run_nn              — Early stopping triggered at epoch 39
13:44:41 | INFO     | utils.run_nn       

In [11]:
nn_output['metrics_df'][[
    'tracking_error', 'information_ratio',
    'correlation', 'sharpe_replica', 'max_dd_replica'
]].round(4)

,tracking_error,information_ratio,correlation,sharpe_replica,max_dd_replica
model,,,,,
MLP_w26_h64x32_l10.0,0.0360,-0.0751,0.8453,0.9003,0.1027
MLP_w52_h64x32_l10.001,0.0518,-0.4214,0.6682,1.0641,0.0525
MLP_w26_h128x64x32_l10.001,0.0449,-0.7430,0.7858,0.6199,0.0683
LSTM_w52_h64_l10.0,0.0363,-0.2999,0.8424,0.8122,0.1003


## 5 · Full Comparison
All models evaluated on their respective test periods.

> **Note on alignment**: linear models and Kalman use a fixed
> `train_end_date='2018-12-31'` cutoff.  The NN uses a 75 % fraction
> split.  If the two test periods differ, either align via a common
> start date or note the difference when comparing metrics.

In [12]:
from utils.evaluation import run_evaluation

all_results = {}

# Linear models
if linear_results:
    all_results.update(linear_results)

# Kalman
if kalman_result is not None:
    all_results['Kalman'] = kalman_result

# NN best config
nn_pkl = PICKLE_DIR / 'nn_results.pkl'
if nn_pkl.exists():
    with open(nn_pkl, 'rb') as fh:
        nn_data = pickle.load(fh)
    all_results['NN_best'] = nn_data['best_result']

print('Models being compared:', list(all_results.keys()))

Models being compared: ['OLS', 'Ridge', 'LASSO', 'ElasticNet', 'WOLS', 'Kalman', 'NN_best']


In [13]:
from pathlib import Path

PICKLE_DIR = ROOT / 'results' / 'data' / 'picklefiles'

print("PICKLE DIR:", PICKLE_DIR.resolve())
print("Exists:", PICKLE_DIR.exists())
print()

for name in ['linear_results.pkl', 'kalman_results.pkl', 'nn_results.pkl']:
    p = PICKLE_DIR / name
    print(f"{name}: {'EXISTS' if p.exists() else 'MISSING'}")

print()
print("linear_results:", linear_results)
print("kalman_result: ", kalman_result)

PICKLE DIR: C:\Users\giuli\Repositories\fintech-group-work\BusinessCase3\results\data\picklefiles
Exists: True

linear_results.pkl: EXISTS
kalman_results.pkl: EXISTS
nn_results.pkl: EXISTS

linear_results: {'OLS': {'replica_returns': 2019-01-01    0.020894
2019-01-08    0.008941
2019-01-15    0.003128
2019-01-22    0.003308
2019-01-29    0.002993
                ...   
2021-03-23   -0.008142
2021-03-30    0.001627
2021-04-06    0.011201
2021-04-13    0.004487
2021-04-20   -0.002035
Name: replica, Length: 121, dtype: float64, 'target_returns': Date
2019-01-01    0.015305
2019-01-08    0.012077
2019-01-15    0.008551
2019-01-22    0.002756
2019-01-29    0.002706
                ...   
2021-03-23   -0.003853
2021-03-30   -0.002620
2021-04-06    0.009555
2021-04-13    0.008686
2021-04-20    0.004717
Name: MonsterIndex, Length: 121, dtype: float64, 'weights_history':             RX1 Comdty  TY1 Comdty  GC1 Comdty  CO1 Comdty  ES1 Comdty  \
2019-01-01   -0.027995    0.130061    0.084712   -0

In [14]:
metrics = run_evaluation(all_results, save_prefix='final')
metrics[[
    'tracking_error', 'information_ratio',
    'correlation', 'sharpe_replica', 'max_dd_replica'
]].round(4)

13:44:56 | INFO     | utils.evaluation          — ============================================================
13:44:56 | INFO     | utils.evaluation          — EVALUATION — START  (7 models)
13:44:56 | INFO     | utils.evaluation          — ============================================================
13:44:56 | INFO     | utils.evaluation          — 
            tracking_error  information_ratio  correlation  sharpe_replica
model                                                                     
OLS                 0.0380            -0.5265       0.8484          1.4805
Ridge               0.0419            -0.2441       0.8116          1.7002
LASSO               0.0414            -0.2548       0.8166          1.6747
ElasticNet          0.0445            -0.2593       0.7860          1.8740
WOLS                0.0375            -0.5155       0.8523          1.4922
Kalman              0.0445            -0.1764       0.7865          1.6832
NN_best             0.0360            -0.0751 

,tracking_error,information_ratio,correlation,sharpe_replica,max_dd_replica
model,,,,,
OLS,0.0380,-0.5265,0.8484,1.4805,0.0819
Ridge,0.0419,-0.2441,0.8116,1.7002,0.0801
LASSO,0.0414,-0.2548,0.8166,1.6747,0.0773
ElasticNet,0.0445,-0.2593,0.7860,1.8740,0.0595
WOLS,0.0375,-0.5155,0.8523,1.4922,0.0822
Kalman,0.0445,-0.1764,0.7865,1.6832,0.0880
NN_best,0.0360,-0.0751,0.8453,0.9003,0.1027


## 6 · Transaction Cost Analysis
Applies one-way cost scenarios of **0, 2, 5, 10 bps** to every model that exposes a `weights_history`.

In [15]:
from utils.transaction_costs import run_transaction_cost_analysis
from utils.evaluation import run_evaluation_with_costs

# ── Align all models to common test start for fair comparison ────────────────
common_start = max(
    res["weights_history"].index[0]
    for res in all_results.values()
    if "weights_history" in res
)
print("Common start:", common_start)

all_results_aligned = {
    name: {
        k: v.loc[common_start:] if isinstance(v, (pd.Series, pd.DataFrame)) else v
        for k, v in res.items()
    }
    for name, res in all_results.items()
}

cost_tables = run_transaction_cost_analysis(
    all_results_aligned,
    cost_scenarios=[0.0, 2.0, 5.0, 10.0],
    save_prefix='tc',
)

cost_tables = run_transaction_cost_analysis(
    all_results,
    cost_scenarios=[0.0, 2.0, 5.0, 10.0],
    save_prefix='tc',
)

Common start: 2019-01-01 00:00:00
13:44:59 | INFO     | utils.transaction_costs   — ============================================================
13:44:59 | INFO     | utils.transaction_costs   — TRANSACTION COST ANALYSIS — START  (7 models)
13:44:59 | INFO     | utils.transaction_costs   — ============================================================
13:44:59 | INFO     | utils.transaction_costs   — Computing cost scenarios for 'OLS' …
13:44:59 | INFO     | utils.transaction_costs   — Computing cost scenarios for 'Ridge' …
13:44:59 | INFO     | utils.transaction_costs   — Computing cost scenarios for 'LASSO' …
13:44:59 | INFO     | utils.transaction_costs   — Computing cost scenarios for 'ElasticNet' …
13:44:59 | INFO     | utils.transaction_costs   — Computing cost scenarios for 'WOLS' …
13:44:59 | INFO     | utils.transaction_costs   — Computing cost scenarios for 'Kalman' …
13:44:59 | INFO     | utils.transaction_costs   — Computing cost scenarios for 'NN_best' …
13:44:59 | INFO     

In [16]:
combined_metrics = run_evaluation_with_costs(
    all_results,
    cost_bps=5.0,
    save_prefix='final_with_costs',
)

combined_metrics[[
    'tracking_error', 'information_ratio',
    'net_te', 'net_ir',
    'sharpe_replica', 'net_sharpe',
]].round(4)

13:45:02 | INFO     | utils.evaluation          — ============================================================
13:45:02 | INFO     | utils.evaluation          — EVALUATION — START  (7 models)
13:45:02 | INFO     | utils.evaluation          — ============================================================
13:45:02 | INFO     | utils.evaluation          — 
            tracking_error  information_ratio  correlation  sharpe_replica
model                                                                     
OLS                 0.0380            -0.5265       0.8484          1.4805
Ridge               0.0419            -0.2441       0.8116          1.7002
LASSO               0.0414            -0.2548       0.8166          1.6747
ElasticNet          0.0445            -0.2593       0.7860          1.8740
WOLS                0.0375            -0.5155       0.8523          1.4922
Kalman              0.0445            -0.1764       0.7865          1.6832
NN_best             0.0360            -0.0751 

,tracking_error,information_ratio,net_te,net_ir,sharpe_replica,net_sharpe
model,,,,,,
OLS,0.0380,-0.5265,0.0380,-1.3284,1.4805,0.9918
Ridge,0.0419,-0.2441,0.0416,-0.4458,1.7002,1.5688
LASSO,0.0414,-0.2548,0.0413,-0.3956,1.6747,1.5855
ElasticNet,0.0445,-0.2593,0.0445,-0.3212,1.8740,1.8266
WOLS,0.0375,-0.5155,0.0376,-1.3445,1.4922,0.9939
Kalman,0.0445,-0.1764,0.0444,-0.1962,1.6832,1.6716
NN_best,0.0360,-0.0751,0.0360,-0.1149,0.9003,0.8763


### Interpretation

Fill this section after running.  Suggested prompts:
- Which model has the lowest **Tracking Error**?  Does it also have the best **IR**?
- Is any model's IR suspiciously positive?  If so, check gross exposure and cumulative active returns.
- At 5 bps, how much does IR degrade per model?  Which model is most turnover-sensitive?
- How does the **Kalman filter** compare to the rolling-window linear models — does adaptive weighting help?
- How does the **NN** compare to Kalman — does extra capacity pay off, or does it overfit the limited time series?
